In [10]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# 1. VẤN ĐỀ 1: TẢI DỮ LIỆU TỪ FILE CÓ SẴN VÀ THÊM THAM SỐ BỎ QUA DÒNG LỖI

print("--- [VẤN ĐỀ 1]: Tải dữ liệu từ file PATIENT_HEART_RATE.CSV và thêm header ---")
# Khai báo đủ 10 cột để khớp với cấu trúc tệp dữ liệu mẫu
column_names = ["Id", "Name", "Age", "Weight", "m0006", "m0612", "m1218", "f0006", "f0612", "f1218"]


df = pd.read_csv("PATIENT_HEART_RATE.CSV", header=None, names=column_names, on_bad_lines='skip')
display(df.head(5))
print("\n" + "="*80 + "\n")


# 2. VẤN ĐỀ 2: TÁCH CỘT TRÙNG NHIỀU THÔNG TIN (NAME -> FIRSTNAME & LASTNAME)
print("--- [VẤN ĐỀ 2]: Tách cột Name thành Firstname và Lastname ---")
df[['Firstname', 'Lastname']] = df['Name'].str.split(pat=' ', n=1, expand=True)
df = df.drop('Name', axis=1)
display(df.head(5))
print("\n" + "="*80 + "\n")


# 3. VẤN ĐỀ 3: KHÔNG THỐNG NHẤT ĐƠN VỊ ĐO LƯỜNG (CHUYỂN LBS VỀ KGS)
print("--- [VẤN ĐỀ 3]: Chuẩn hóa đơn vị cột Weight về kgs ---")
weight = df['Weight'].copy()
for i in range(0, len(weight)):
    x = str(weight.iloc[i])
    if "lbs" in x[-3:]:
        x = x[:-3]         
        float_x = float(x)
        y = int(float_x / 2.2) 
        y = str(y) + "kgs"
        weight.iloc[i] = y
df['Weight'] = weight
display(df.head(5))
print("\n" + "="*80 + "\n")

# 4. VẤN ĐỀ 4: XUẤT HIỆN DÒNG DỮ LIỆU RỖNG HOÀN TOÀN
print("--- [VẤN ĐỀ 4]: Xóa bỏ các dòng rỗng hoàn toàn ---")
df.dropna(how="all", inplace=True)
display(df.head(5))
print("\n" + "="*80 + "\n")

# 5. VẤN ĐỀ 5: TRÙNG LẶP THÔNG TIN HOÀN TOÀN (DROP DUPLICATES)

print("--- [VẤN ĐỀ 5]: Loại bỏ các dòng trùng lặp thông tin hoàn toàn ---")
df = df.drop_duplicates(subset=['Firstname', 'Lastname', 'Age', 'Weight'])
display(df)
print("\n" + "="*80 + "\n")


# 6. VẤN ĐỀ 6: DỮ LIỆU BỊ ẢNH HƯỞNG BỞI LỖI NON-ASCII

print("--- [VẤN ĐỀ 6]: Xử lý ký tự lạ non-ASCII (Sửa triệt để chữ Préféré) ---")
df['Firstname'] = df['Firstname'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
df['Lastname'] = df['Lastname'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
display(df)
print("\n" + "="*80 + "\n")


# 7. VẤN ĐỀ 7: KHẢO SÁT VÀ XỬ LÝ DỮ LIỆU THIẾU TRÊN AGE VÀ WEIGHT
print("--- [VẤN ĐỀ 7]: Xử lý dữ liệu thiếu trên biến Age và Weight chuẩn theo đề bài ---")
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['Weight'] = df['Weight'].astype(str).str.replace('kgs', '', regex=False)
df['Weight'] = pd.to_numeric(df['Weight'], errors='coerce')

# Thiếu cả hai (Age và Weight đều NaN) -> Xóa dòng
df.dropna(subset=['Age', 'Weight'], how='all', inplace=True)

# Thiếu một trong hai -> Điền giá trị trung bình (Mean)
mean_age = df['Age'].mean()
df['Age'] = df['Age'].fillna(mean_age)

mean_weight = df['Weight'].mean()
df['Weight'] = df['Weight'].fillna(mean_weight)
df['Weight'] = df['Weight'].round(1)

display(df)
print("\n" + "="*80 + "\n")


# =========================================================================
# 8. VẤN ĐỀ 8: MỘT CỘT CHỨA QUÁ NHIỀU THÔNG TIN CẦN PHÂN RÃ (MELT)
# =========================================================================
print("--- [VẤN ĐỀ 8]: Dùng melt để đưa các cột nhịp tim hỗn hợp về cấu trúc chuẩn ---")
df_melted = pd.melt(df, id_vars=['Id', 'Age', 'Weight', 'Firstname', 'Lastname'], 
                    value_name="PulseRate", var_name="sex_and_time")
df_melted = df_melted.sort_values(by=['Id', 'Age', 'Weight', 'Firstname', 'Lastname'])
display(df_melted.head(10))
print("\n" + "="*80 + "\n")


# 9 & 10. VẤN ĐỀ 9 & 10: TÁCH CỘT SEX_AND_TIME THÀNH 2 BIẾN "SEX" VÀ "TIME"

print("--- [VẤN ĐỀ 9 & 10]: Tách chuỗi thời gian và giới tính ra làm 2 cột độc lập ---")
tmp_df = df_melted['sex_and_time'].str.extract(r'(?P<Sex>[amf])(?P<hours_lower>\d{2})(?P<hours_upper>\d{2})', expand=True)
tmp_df['Time'] = tmp_df['hours_lower'] + "-" + tmp_df['hours_upper']

df_cleaned = pd.concat([df_melted, tmp_df], axis=1)
df_cleaned = df_cleaned.drop(['sex_and_time', 'hours_lower', 'hours_upper'], axis=1)
display(df_cleaned.head(10))
print("\n" + "="*80 + "\n")



# 11. VẤN ĐỀ 11: KHẢO SÁT VÀ XỬ LÝ DỮ LIỆU THIẾU TRÊN BIẾN HUYẾT ÁP/NHỊP TIM

print("--- [VẤN ĐỀ 11]: Làm sạch các ký tự đại diện thiếu '-' ở cột nhịp tim ---")
df_cleaned['PulseRate'] = df_cleaned['PulseRate'].replace('-', np.nan)
df_cleaned['PulseRate'] = pd.to_numeric(df_cleaned['PulseRate'], errors='coerce')
df_cleaned.dropna(subset=['PulseRate'], inplace=True)
display(df_cleaned.head(10))
print("\n" + "="*80 + "\n")


# 
# 12. VẤN ĐỀ 12: TỔNG KẾT, RÚT GỌN INDEX VÀ XUẤT RA FILE PATIENT_HEART_RATE_CLEAN.CSV

print("--- [VẤN ĐỀ 12]: Sắp xếp lại chỉ mục index và xuất file sạch thành công ---")
df_cleaned['Weight'] = df_cleaned['Weight'].astype(str) + "kgs"
df_cleaned = df_cleaned.reset_index(drop=True)

# Xuất dữ liệu sạch ra file kết quả cuối cùng
df_cleaned.to_csv('patient_heart_rate_clean.csv', index=False)
print(" xuất file 'patient_heart_rate_clean.csv'!")
display(df_cleaned)

--- [VẤN ĐỀ 1]: Tải dữ liệu từ file PATIENT_HEART_RATE.CSV và thêm header ---


,Id,Name,Age,Weight,m0006,m0612,m1218,f0006,f0612,f1218
0,1.0,Mickéy Mousé,56.0,70kgs,72,69,71,-,-,-
1,2.0,Donald Duck,34.0,154.89lbs,-,-,-,85,84,76
2,3.0,Mini Mouse,16.0,NaN,-,-,-,65,69,72
3,4.0,Scrooge McDuck,NaN,78kgs,78,79,72,-,-,-
4,5.0,Pink Panther,54.0,198.658lbs,-,-,-,69,NaN,75




--- [VẤN ĐỀ 2]: Tách cột Name thành Firstname và Lastname ---


,Id,Age,Weight,m0006,m0612,m1218,f0006,f0612,f1218,Firstname,Lastname
0,1.0,56.0,70kgs,72,69,71,-,-,-,Mickéy,Mousé
1,2.0,34.0,154.89lbs,-,-,-,85,84,76,Donald,Duck
2,3.0,16.0,NaN,-,-,-,65,69,72,Mini,Mouse
3,4.0,NaN,78kgs,78,79,72,-,-,-,Scrooge,McDuck
4,5.0,54.0,198.658lbs,-,-,-,69,NaN,75,Pink,Panther




--- [VẤN ĐỀ 3]: Chuẩn hóa đơn vị cột Weight về kgs ---


...

,Id,Age,Weight,Firstname,Lastname,PulseRate,Sex,Time
0,1.0,56.0,70.0,Mickey,Mouse,72,m,00-06
12,1.0,56.0,70.0,Mickey,Mouse,69,m,06-12
24,1.0,56.0,70.0,Mickey,Mouse,71,m,12-18
36,1.0,56.0,70.0,Mickey,Mouse,-,f,00-06
48,1.0,56.0,70.0,Mickey,Mouse,-,f,06-12
60,1.0,56.0,70.0,Mickey,Mouse,-,f,12-18
1,2.0,34.0,70.0,Donald,Duck,-,m,00-06
13,2.0,34.0,70.0,Donald,Duck,-,m,06-12
25,2.0,34.0,70.0,Donald,Duck,-,m,12-18
37,2.0,34.0,70.0,Donald,Duck,85,f,00-06




--- [VẤN ĐỀ 11]: Làm sạch các ký tự đại diện thiếu '-' ở cột nhịp tim ---


,Id,Age,Weight,Firstname,Lastname,PulseRate,Sex,Time
0,1.0,56.0,70.0,Mickey,Mouse,72.0,m,00-06
12,1.0,56.0,70.0,Mickey,Mouse,69.0,m,06-12
24,1.0,56.0,70.0,Mickey,Mouse,71.0,m,12-18
37,2.0,34.0,70.0,Donald,Duck,85.0,f,00-06
49,2.0,34.0,70.0,Donald,Duck,84.0,f,06-12
61,2.0,34.0,70.0,Donald,Duck,76.0,f,12-18
38,3.0,16.0,71.3,Mini,Mouse,65.0,f,00-06
50,3.0,16.0,71.3,Mini,Mouse,69.0,f,06-12
62,3.0,16.0,71.3,Mini,Mouse,72.0,f,12-18
3,4.0,36.1,78.0,Scrooge,McDuck,78.0,m,00-06




--- [VẤN ĐỀ 12]: Sắp xếp lại chỉ mục index và xuất file sạch thành công ---
 xuất file 'patient_heart_rate_clean.csv'!


,Id,Age,Weight,Firstname,Lastname,PulseRate,Sex,Time
0,1.0,56.0,70.0kgs,Mickey,Mouse,72.0,m,00-06
1,1.0,56.0,70.0kgs,Mickey,Mouse,69.0,m,06-12
2,1.0,56.0,70.0kgs,Mickey,Mouse,71.0,m,12-18
3,2.0,34.0,70.0kgs,Donald,Duck,85.0,f,00-06
4,2.0,34.0,70.0kgs,Donald,Duck,84.0,f,06-12
5,2.0,34.0,70.0kgs,Donald,Duck,76.0,f,12-18
6,3.0,16.0,71.3kgs,Mini,Mouse,65.0,f,00-06
7,3.0,16.0,71.3kgs,Mini,Mouse,69.0,f,06-12
8,3.0,16.0,71.3kgs,Mini,Mouse,72.0,f,12-18
9,4.0,36.1,78.0kgs,Scrooge,McDuck,78.0,m,00-06
